# `dtype` konvertieren

Manchmal passen die pandas-Datentypen nicht wirklich gut. Dies kann z.B. auf Serialisierungsformate zurückzuführen sein, die keine Typinformationen enthalten. Manchmal solltet ihr jedoch den Typ auch ändern, um eine bessere Performance zu erzielen – entweder  mehr Manipulationsmöglichkeiten oder weniger Speicherbedarf. In den folgenden Beispielen werden wir verschiedene Konvertierungen einer `Series` vornehmen:

In [1]:
import numpy as np
import pandas as pd

In [2]:
s = pd.Series(np.random.randn(7))

In [3]:
s

0   -1.082608
1   -2.924034
2   -0.501255
3   -0.040184
4    1.050727
5    0.584097
6    1.402607
dtype: float64

## Automatische Konvertierung

[pandas.Series.convert_dtypes](https://pandas.pydata.org/docs/reference/api/pandas.Series.convert_dtypes.html) versucht, eine `Series` in einen Typ zu konvertieren, der `NA` unterstützt. Im Fall unserer `Series` wird der Typ von `float64` in `Float64` geändert:

In [4]:
s.convert_dtypes()

0   -1.082608
1   -2.924034
2   -0.501255
3   -0.040184
4    1.050727
5    0.584097
6    1.402607
dtype: Float64

Bedauerlicherweise habe ich jedoch mit `convert_dtypes` kaum Kontrolle darüber, in welchen Datentyp konvertiert wird. Daher bevorzuge ich [pandas.Series.astype](https://pandas.pydata.org/docs/reference/api/pandas.Series.astype.html):

In [5]:
s.astype("float32")

0   -1.082608
1   -2.924034
2   -0.501255
3   -0.040184
4    1.050727
5    0.584097
6    1.402607
dtype: float32

Sofern jedoch nicht konvertierbare Werte enthalten sind, wird ein Fehler ausgegeben:

In [6]:
n = pd.Series([np.random.randint(127), np.nan, np.random.randint(127)])
n

0    116.0
1      NaN
2    109.0
dtype: float64

In [7]:
n.astype('int8')

IntCastingNaNError: Cannot convert non-finite values (NA or inf) to integer

Fehler, wie dieser `IntCastingNaNError` können vermieden werden, indem durch `errors = "ignore"` ggf. der ursprüngliche Datentyp beibehalten wird:

In [8]:
n.astype('int8', errors="ignore")

0    116.0
1      NaN
2    109.0
dtype: float64

Die Verwendung des richtigen Typs kann Speicherplatz einsparen. Üblich ist ein 8 Byte breiter Datentyp, also `int64` oder `float64`. Wenn ihr einen schmaleren Typ verwenden könnt, reduziert dies den Speicherverbrauch deutlich, sodass ihr mehr Daten verarbeiten könnt. Ihr könnt NumPy verwenden, um die Grenzen von Integer- und Float-Typen zu überprüfen:

In [9]:
np.iinfo("int8")

iinfo(min=-128, max=127, dtype=int8)

In [10]:
np.iinfo("int64")

iinfo(min=-9223372036854775808, max=9223372036854775807, dtype=int64)

In [11]:
np.finfo("float32")

finfo(resolution=1e-06, min=-3.4028235e+38, max=3.4028235e+38, dtype=float32)

In [12]:
np.finfo('float64')

finfo(resolution=1e-15, min=-1.7976931348623157e+308, max=1.7976931348623157e+308, dtype=float64)

## Speicherverbrauch

Um den Speicherverbrauch der `Series` zu berechnen, könnt ihr [pandas.Series.nbytes](https://pandas.pydata.org/docs/reference/api/pandas.Series.nbytes.html) verwenden um den Speicher, der von den Daten verwendet wird, zu ermitteln. [pandas.Series.memory_usage](https://pandas.pydata.org/docs/reference/api/pandas.Series.memory_usage.html) erfasst darüberhinaus auch den Indexspeicher und den Datentyp. Mit `deep=True` lässt sich auch der Speicherverbrauch auf Systemebene ermitteln.

In [13]:
s.nbytes

56

In [14]:
s.astype("Float32").nbytes

35

In [15]:
s.memory_usage()

188

In [16]:
s.astype("Float32").memory_usage()

167

In [17]:
s.memory_usage(deep=True)

188

## String- und Kategorietypen

Die Methode [pandas.Series.astype](https://pandas.pydata.org/docs/reference/api/pandas.Series.astype.html) kann auch numerische Reihen in Zeichenketten umwandeln, wenn ihr `str` übergebt. Beachtet den `dtype` im folgenden Beispiel:

In [18]:
s.astype(str)

0      -1.082608205271088
1     -2.9240339130190485
2     -0.5012551265145538
3    -0.04018417684856264
4       1.050726685768491
5       0.584097344603143
6      1.4026068664869746
dtype: object

In [19]:
s.astype(str).memory_usage()

188

In [20]:
s.astype(str).memory_usage(deep=True)

603

Zur Konvertierung in einen kategorialen Typ könnt ihr `'category'` als Typ übergeben:

In [21]:
s.astype(str).astype("category")

0      -1.082608205271088
1     -2.9240339130190485
2     -0.5012551265145538
3    -0.04018417684856264
4       1.050726685768491
5       0.584097344603143
6      1.4026068664869746
dtype: category
Categories (7, object): ['-0.04018417684856264', '-0.5012551265145538', '-1.082608205271088', '-2.9240339130190485', '0.584097344603143', '1.050726685768491', '1.4026068664869746']

Eine kategoriale `Series` ist nützlich für String-Daten und kann zu großen Speichereinsparungen führen. Das liegt daran, dass pandas bei der Konvertierung in kategoriale Daten nicht länger Python-Strings für jeden Wert verwendet, sondern sich wiederholende Werte nicht dupliziert werden. Ihr habt immer noch alle Funktionen des `str`-Attributs, aber ihr spart viel Speicherplatz wenn ihr viele doppelte Werte habt und steigert die Leistung, da ihr nicht so viele String-Operationen durchführen müsst.

In [22]:
s.astype("category").memory_usage(deep=True)

495

## Geordnete Kategorien

Um geordnete Kategorien zu erstellen, müsst ihr einen eigenen [pandas.CategoricalDtype](https://pandas.pydata.org/docs/reference/api/pandas.CategoricalDtype.html) definieren:

In [23]:
from pandas.api.types import CategoricalDtype


sorted = pd.Series(sorted(set(s)))
cat_dtype = CategoricalDtype(categories=sorted, ordered=True)

s.astype(cat_dtype)

0   -1.082608
1   -2.924034
2   -0.501255
3   -0.040184
4    1.050727
5    0.584097
6    1.402607
dtype: category
Categories (7, float64): [-2.924034 < -1.082608 < -0.501255 < -0.040184 < 0.584097 < 1.050727 < 1.402607]

In [24]:
s.astype(cat_dtype).memory_usage(deep=True)

495

In der folgenden Tabelle sind die Typen aufgeführt, die ihr an `astype` übergeben könnt.

Datentyp | Beschreibung
:------- | :-----------
`str`, `'str'` | in Python-String konvertieren
`'string'` | in Pandas-String konvertieren mit `pandas.NA`
`int`, `'int'`, `'int64'` | in NumPy `int64` konvertieren
`'int32'`, `'uint32'` | in NumPy `int32` konvertieren
`'Int64'` | in pandas `Int64` konvertieren mit `pandas.NA`
`float`, `'float'`, `'float64'` | in Floats konvertieren
`'category'` | in `CategoricalDtype` konvertieren mit `pandas.NA`

## Umwandlung in andere Datentypen

Die Methode [pandas.Series.to_numpy](https://pandas.pydata.org/docs/reference/api/pandas.Series.to_numpy.html) oder die Eigenschaft [pandas.Series.values](https://pandas.pydata.org/docs/reference/api/pandas.Series.values.html) liefert uns ein NumPy-Array mit Werten, und [pandas.Series.to_list](https://pandas.pydata.org/docs/reference/api/pandas.Series.to_list.html) gibt eine Python-Liste mit Werten zurück. Warum solltet ihr das wollen? pandas-Objekte sind meist viel benutzerfreundlicher und der Code lässt sich leichter lesen. Zudem werden Python-Listen sehr viel langsamer verarbeitet werden können. Mit [pandas.Series.to_frame](https://pandas.pydata.org/docs/reference/api/pandas.Series.to_frame.html) könnt ihr ggf. einen DataFrame mit einer einzigen Spalte erzeugen:

In [25]:
s.to_frame()

,0
0,-1.082608
1,-2.924034
2,-0.501255
3,-0.040184
4,1.050727
5,0.584097
6,1.402607


Auch die Funktion [pandas.to_datetime](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html) kann hilfreich sein um in pandas um Werte in Datum und Uhrzeit zu konvertieren.